# Финальная оценка на hold-out
**Правило:** этот ноутбук запускается один раз, и ни одно решение не меняется по его результатам.
Все выборы (признаки, параметры, порог, отсутствие калибровки) сделаны на dev.

Что считаем:
0. Какие признаки выдают пол (на dev, hold-out не используется).
1. Качество моделей на hold-out с доверительными интервалами (bootstrap).
2. Калибровка финальной модели на hold-out.
3. Решения по порогу 1/(1+R) для разных R, сравнение с dev.
4. Результаты по полу.
5. Причины отказа у отказанных клиентов.

Нужны: models/lgbm_explainable.joblib, features_explainable.json (09), lgbm_dev.joblib (07),
data/processed/oof_explainable.parquet (10).

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

sys.path.insert(0, "..")
from src.data import load_features, split_dev_holdout
from src.explain import GROUPS, MIN_REASON_SHAP, group_shap, top_reasons
from src.features import ID_COL, TARGET_COL
from src.metrics import calibration_report, calibration_table, ks_stat
from src.models import RANDOM_STATE, make_lgbm, make_logreg, to_lgbm

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)
pd.set_option("display.width", 140)

df = load_features(PROCESSED / "train_features.parquet")
dev, hold = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")
features = json.loads((MODELS / "features_explainable.json").read_text())
y_h = hold[TARGET_COL].values
print(f"dev: {len(dev)} | hold-out: {len(hold)} | доля дефолтов: dev {dev[TARGET_COL].mean():.4f}, "
      f"hold-out {y_h.mean():.4f}")

## 0. Какие признаки выдают пол (только dev)
Модель восстанавливает пол по объяснимым признакам с AUC 0.887. Смотрим, за счёт чего.

In [ ]:
sub = dev[dev["CODE_GENDER"].notna()].sample(100_000, random_state=RANDOM_STATE)
yg = (sub["CODE_GENDER"] == "F").astype(int).values
Xg = to_lgbm(sub[features])
mg = make_lgbm(n_estimators=200, learning_rate=0.1, importance_type="gain").fit(Xg, yg)
imp = pd.Series(mg.feature_importances_, index=Xg.columns)
print("Топ-12 признаков, по которым восстанавливается пол (% суммарного gain):")
print((imp / imp.sum() * 100).round(1).sort_values(ascending=False).head(12))

## 1. Качество на hold-out
Три модели: финальная (LightGBM, объяснимые признаки), LightGBM на полном наборе (чтобы
увидеть цену принципа ещё раз) и логрег на объяснимых признаках.

In [ ]:
model_e = joblib.load(MODELS / "lgbm_explainable.joblib")
model_full = joblib.load(MODELS / "lgbm_dev.joblib")

X_e = to_lgbm(hold[features])
X_full = to_lgbm(hold.drop(columns=[TARGET_COL, ID_COL]))

lr = make_logreg(dev[features]).fit(dev[features], dev[TARGET_COL])

preds = {
    "LightGBM, только объяснимые (финальная)": model_e.predict_proba(X_e)[:, 1],
    "LightGBM, полный набор": model_full.predict_proba(X_full)[:, 1],
    "LogReg, только объяснимые": lr.predict_proba(hold[features])[:, 1],
}

In [ ]:
rows = []
for name, p in preds.items():
    auc = roc_auc_score(y_h, p)
    rows.append({"model": name, "ROC AUC": round(auc, 4), "Gini": round(2 * auc - 1, 4),
                 "PR AUC": round(average_precision_score(y_h, p), 4),
                 "KS": round(ks_stat(y_h, p), 4), "Brier": round(brier_score_loss(y_h, p), 4)})
metrics = pd.DataFrame(rows).set_index("model")
metrics

In [ ]:
# Bootstrap: доверительные интервалы AUC и попарные разницы на одних и тех же ресэмплах
rng = np.random.default_rng(RANDOM_STATE)
names = list(preds)
n, B = len(y_h), 500
boot = {k: [] for k in names}
for _ in range(B):
    idx = rng.integers(0, n, n)
    for k in names:
        boot[k].append(roc_auc_score(y_h[idx], preds[k][idx]))
boot = {k: np.array(v) for k, v in boot.items()}

for k in names:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    print(f"{k}: AUC {roc_auc_score(y_h, preds[k]):.4f}, 95% ДИ [{lo:.4f}; {hi:.4f}]")

print()
pairs = [(names[1], names[0]), (names[0], names[2])]
for a, b in pairs:
    d = boot[a] - boot[b]
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"Разница AUC ({a}) - ({b}): {d.mean():+.4f}, 95% ДИ [{lo:+.4f}; {hi:+.4f}]")

## 2. Калибровка финальной модели на hold-out

In [ ]:
p_final = preds[names[0]]
print(pd.DataFrame([calibration_report("hold-out", y_h, p_final)]).set_index("model"))
calibration_table(y_h, p_final, bins=10).round(4)

## 3. Решения по порогу 1/(1+R)
Порог берём теоретический, не подбирая на hold-out. Для сравнения те же расчёты на OOF dev.

In [ ]:
oof = pd.read_parquet(PROCESSED / "oof_explainable.parquet")


def decide(y, p, R):
    t = 1 / (1 + R)
    reject = p >= t
    fn = int(((~reject) & (y == 1)).sum())
    fp = int((reject & (y == 0)).sum())
    n_ = len(y)
    naive = min(R * y.mean(), 1 - y.mean())
    loss = (R * fn + fp) / n_
    return {
        "R": R, "порог": round(t, 3),
        "доля одобрений": round(1 - reject.mean(), 3),
        "отсеяно дефолтов": round(((reject) & (y == 1)).sum() / (y == 1).sum(), 3),
        "дефолт среди одобренных": round(fn / max((~reject).sum(), 1), 4),
        "потери на клиента": round(loss, 3),
        "экономия, %": round((1 - loss / naive) * 100, 1),
    }


res = pd.DataFrame(
    [{"выборка": "hold-out", **decide(y_h, p_final, R)} for R in [3, 5, 8, 10]]
    + [{"выборка": "dev (OOF)", **decide(oof[TARGET_COL].values, oof["baseline"].values, R)} for R in [3, 5, 8, 10]]
).set_index(["R", "выборка"]).sort_index()
res

## 4. Результаты по полу (порог R = 5)

In [ ]:
R = 5
t = 1 / (1 + R)
gender = hold["CODE_GENDER"].values
rows = []
for g in ["F", "M"]:
    m = gender == g
    rows.append({
        "пол": g, "клиентов": int(m.sum()), "observed": y_h[m].mean(), "predicted": p_final[m].mean(),
        "AUC в группе": roc_auc_score(y_h[m], p_final[m]), "отказов, %": (p_final[m] >= t).mean() * 100,
    })
fair = pd.DataFrame(rows).set_index("пол").round(4)
print(fair)
print(f"\nОтношение долей отказов M / F: {fair.loc['M', 'отказов, %'] / fair.loc['F', 'отказов, %']:.2f}")

## 5. Причины отказа у отказанных клиентов (R = 5)

In [ ]:
contrib = model_e.booster_.predict(X_e, pred_contrib=True)
sv = contrib[:, :-1]
grouped = group_shap(sv, X_e.columns)
reasons = top_reasons(grouped, k=3, min_shap=MIN_REASON_SHAP)

refused = np.where(p_final >= t)[0]
n_reasons = pd.Series([len(reasons[i]) for i in refused]).value_counts().sort_index()
print(f"Отказано: {len(refused)} клиентов ({len(refused) / len(p_final):.1%})")
print("Сколько причин показано (порог вклада "
      f"{MIN_REASON_SHAP}):")
print(n_reasons.rename("клиентов"))

In [ ]:
counts = pd.Series([text for i in refused for text, _ in reasons[i]]).value_counts()
print("Частота причин среди отказанных, %:")
print((counts / len(refused) * 100).round(1))

for k in range(3):
    vals = np.array([reasons[i][k][1] for i in refused if len(reasons[i]) > k])
    print(f"Причина №{k + 1}: квантили вклада 10% / 50% / 90%: {np.quantile(vals, [0.1, 0.5, 0.9]).round(2)}")

In [ ]:
# Примеры
for i in refused[np.argsort(-p_final[refused])][[0, len(refused) // 10, len(refused) // 2]]:
    print(f"--- вероятность дефолта {p_final[i]:.1%} | факт: {y_h[i]} ---")
    for text, val in reasons[i]:
        print(f"  {text}  (вклад {val:+.2f})")
    print()

## 6. Сохраняем итог и конфигурацию решения для бота

In [ ]:
report = {
    "holdout_rows": int(len(hold)),
    "holdout_default_rate": round(float(y_h.mean()), 4),
    "metrics": metrics.reset_index().to_dict(orient="records"),
}
(REPORTS / "holdout_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))

decision_config = {"cost_ratio": 5, "threshold": round(1 / 6, 4), "min_reason_shap": MIN_REASON_SHAP,
                   "calibrated": False, "model": "models/lgbm_explainable.joblib",
                   "features": "models/features_explainable.json"}
(MODELS / "decision_config.json").write_text(json.dumps(decision_config, ensure_ascii=False, indent=2))
print("Сохранено: reports/holdout_report.json, models/decision_config.json")